In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup

In [2]:
KEYWORDS = {
    "Food/Pantry":            ["food pantry", "food bank", "meals", "soup kitchen",
                               "nutrition", "groceries", "hunger"],
    "Housing":                ["housing", "shelter", "homeless", "rental assistance",
                               "affordable housing", "eviction", "homeownership"],
    "Health/Mental Health":   ["mental health", "behavioral health", "counseling",
                               "therapy", "clinic", "medical", "healthcare",
                               "substance", "recovery", "addiction"],
    "Education/Tutoring":     ["tutoring", "after school", "afterschool", "ged",
                               "esol", "esl", "adult education", "literacy",
                               "scholarship", "college", "mentoring"],
    "Youth/Childcare":        ["youth", "childcare", "child care", "early childhood",
                               "daycare", "preschool", "summer camp"],
    "Workforce/Jobs":         ["workforce", "job training", "employment", "career",
                               "financial literacy", "entrepreneur", "small business"],
    "Legal/Immigration":      ["legal", "immigration", "citizenship", "attorney",
                               "asylum", "deportation"],
    "Senior/Disability":      ["senior", "elder", "aging", "disability",
                               "disabilities", "adult day", "early intervention"],
    "Arts/Recreation":        ["arts", "music", "theater", "gallery", "athletics",
                               "sports", "recreation", "squash", "basketball", "soccer"],
    "Domestic Violence/Crisis": ["domestic violence", "crisis", "abuse",
                                 "survivor", "safety planning"],
    "Clothing/Basic Needs":   ["clothing", "diapers", "furniture", "thrift",
                               "basic needs", "toiletries"],
}


# Words that, if ALREADY in the CSV's Services cell, mean we should NOT
#    flag that category. (This is what prevents false "missing" flags when
#    the CSV uses a different word for the same thing.)
ALREADY_LISTED_KEYWORDS = {
    "Food/Pantry": ["food", "meal", "pantry", "hunger", "nutrition"],
    "Housing": ["housing", "homeless", "shelter", "eviction"],
    "Health/Mental Health": ["health", "mental", "behavioral", "counsel", "therap",
                             "clinic", "medical", "recovery", "substance", "addiction"],
    "Education/Tutoring": ["education", "tutor", "literacy", "esol", "esl", "ged",
                           "academic", "scholarship", "college", "mentor", "school"],
    "Youth/Childcare": ["youth", "child", "early", "daycare", "preschool", "camp",
                        "after school", "afterschool"],
    "Workforce/Jobs": ["workforce", "job", "employment", "career", "financial",
                       "entrepreneur", "business", "economic"],
    "Legal/Immigration": ["legal", "immigration", "citizenship"],
    "Senior/Disability": ["senior", "elder", "aging", "disab", "adult day",
                          "intervention"],
    "Arts/Recreation": ["art", "music", "theater", "athletic", "sport", "squash",
                        "basketball", "soccer", "recreation"],
    "Domestic Violence/Crisis": ["domestic", "crisis", "abuse", "survivor", "violence"],
    "Clothing/Basic Needs": ["cloth", "diaper", "furniture", "thrift", "basic need"],
}

In [7]:
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                         "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120 Safari/537.36"}

In [8]:
df = pd.read_csv("GWIorgs_v4.csv", dtype=str).fillna("")
df = df[df["Name"].str.strip() != ""].reset_index(drop=True)

report = [] # will hold one row of results per organization

In [9]:
for i in range(len(df)):
    name = df.loc[i,"Name"]
    url = df.loc[i, "URL"]
    listed_services = df.loc[i, "Services"].lower()
    
    # try to download the website's text
    status = "OK"
    website_text = ""
    
    if not url.startswith("http"):
        status = "NO_VALID_URL"
    else:
        try:
            page = requests.get(url, headers=HEADERS, timeout=15)
            if page.status_code != 200:
                status = f"HTTP_{page.status_code}"
            else:
                soup = BeautifulSoup(page.text, "html_parser")
                for tag in soup(["script", "style", "noscript"]):
                    tag.extract()
                website_text = soup.get_text(" ").lower()
        except Exception as e:
            status = f"Error: {type(e).__name__}"
            
    if status != "OK":
        report.append({"Name": name, "URL": url, "Status": status,
                       "PossibleMissing": "",
                       "Note": "Could not read site - check by hand"})
        continue

In [13]:
#compare: website mentions it, but CSV doesn't already list it
missing = []
for category, words in KEYWORDS.items():
    website_mentions_it = any(word in website_text for word in words)
    csv_already_has_it = any(w in listed_services
                            for w in ALREADY_LISTED_KEYWORDS[category])
    if website_mentions_it and not csv_already_has_it:
        missing.append(category)

report.append({"Name": name, "URL": url, "Status": status,
                   "PossibleMissing": "; ".join(missing),
                   "Note": "Review flagged categories against the site"})

In [14]:
#save the results
pd.DataFrame(report).to_csv("service_gap_report.csv", index=False)

ok = sum(1 for r in report if r["Status"] == "OK")
print(f"\nDone. {len(report)} orgs processed. OK: {ok}, {len(report) - ok} could not be read.")
print("Results: service_gap_report.csv")


Done. 67 orgs processed. OK: 0, 67 could not be read.
Results: service_gap_report.csv
